# Globe3D Demo: Fibonacci Sphere with Topography

This notebook demonstrates how to use the `globe3d` library to create a 3D printable globe with exaggerated topography.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import cmocean
import matplotlib.pyplot as plt
from globe3d import (
    generate_sphere_points_fibonacci,
    list_netcdf_variables,
    load_netcdf_grid,
    displace_vertices,
    assign_vertex_colors,
    resize_globe,
    invert_chirality,
    combine_subtractive_globes,
    compute_scale_factor,
    write_obj_with_vertex_colors,
    write_stl_binary,
    plot_vertex_distribution
)

## 1. Generate Reference Sphere

In [ ]:
n_points = 100000
radius = 6371.
print("Generating Fibonacci sphere...")
vertices, faces = generate_sphere_points_fibonacci(n_points, radius)
plot_vertex_distribution(vertices, title='Fibonacci Sphere Distribution')

## 2. Load Topography Data

In [ ]:
# Update this path to your NetCDF file
topogrid = r"F:\no_backup\gmt_grids\ETOPO_2022_v1_60s_N90W180_surface.nc"
try:
    print("Variables:", list_netcdf_variables(topogrid))
    lats, lons, grid = load_netcdf_grid(topogrid, lat_var='lat', lon_var='lon', data_var='z')
except FileNotFoundError:
    print("Grid file not found. Please update the path.")
    # Create dummy data for demonstration if file not found
    lats = np.linspace(-90, 90, 180)
    lons = np.linspace(-180, 180, 360)
    grid = np.zeros((180, 360))
    # Add some features
    grid[90, 180] = 8000 # Everest-ish
    grid[45, 90] = -10000 # Mariana-ish

## 3. Displace Vertices

In [ ]:
vert_exagg = 50
scale = 0.001 * vert_exagg

vertices = displace_vertices(vertices, lats, lons, grid, scale, show_progress=True)

## 4. Assign Colors

In [ ]:
colourmap = cmocean.cm.topo
colours = assign_vertex_colors(vertices, lats, lons, grid, colormap=colourmap, vmin=-8000, vmax=8000, show_progress=True)

## 5. Hollow the Globe

In [ ]:
inner_vertices = resize_globe(vertices, 0.85)
combined_vertices, combined_faces, combined_colors = combine_subtractive_globes(
    vertices, faces, inner_vertices, faces, outer_colors=colours, inner_colors=colours
)

## 6. Export

In [ ]:
scale_factor = compute_scale_factor(combined_vertices, 80)
scaled_vertices = resize_globe(combined_vertices, scale_factor)

objfile = f'topoglobe_{vert_exagg}x_exaggeration_coloured_hollow.obj'
write_obj_with_vertex_colors(objfile, scaled_vertices, combined_faces, combined_colors)
print(f"Saved {objfile}")

stlfile = f'topoglobe_{vert_exagg}x_exaggeration_hollow.stl'
write_stl_binary(stlfile, scaled_vertices, combined_faces)
print(f"Saved {stlfile}")